# Chapter 13: Resource Calculator & Reference Guide

This notebook contains core code snippets and reference links from Chapter 13 Appendices.

**Book Code Repository:** https://rewriting.ai/bookcode/slm.php


---
## 1. GPU Hours & Cost Estimation Calculator

In [ ]:
def calculate_training_time(model_params, tokens, epochs=1, gpu_tflops=312, mfu=0.35):
    """
    Calculate training time for a language model.

    Args:
        model_params: Number of model parameters (e.g., 26e6 for 26M)
        tokens: Number of training tokens (e.g., 100e6 for 100M)
        epochs: Number of training epochs
        gpu_tflops: GPU theoretical peak TFLOPs (e.g., 312 for A100)
        mfu: Model FLOPs Utilization (typically 0.3-0.4)

    Returns:
        dict: Training time in various units
    """
    # Calculate total FLOPs
    total_flops = 6 * model_params * tokens * epochs

    # Calculate actual TFLOPs considering MFU
    actual_tflops = gpu_tflops * mfu

    # Calculate training time
    training_seconds = total_flops / (actual_tflops * 1e12)
    training_hours = training_seconds / 3600
    training_days = training_hours / 24

    return {
        'total_flops': total_flops,
        'actual_tflops': actual_tflops,
        'seconds': training_seconds,
        'minutes': training_seconds / 60,
        'hours': training_hours,
        'days': training_days
    }

# Example 1: MiniMind-26M with 100M tokens
print("Example 1: MiniMind-26M (100M tokens)")
result = calculate_training_time(
    model_params=26e6,
    tokens=100e6,
    gpu_tflops=125,  # V100
    mfu=0.35
)
print(f"Training time: {result['minutes']:.1f} minutes ({result['hours']:.2f} hours)")
print(f"For 1B tokens: {result['hours'] * 10:.1f} hours")
print()

# Example 2: MiniMind-109M with 500M tokens
print("Example 2: MiniMind-109M (500M tokens)")
result = calculate_training_time(
    model_params=109e6,
    tokens=500e6,
    gpu_tflops=312,  # A100
    mfu=0.40
)
print(f"Training time: {result['hours']:.1f} hours ({result['days']:.1f} days)")
print()

# Custom calculation
print("Custom Calculation:")
print("-" * 50)
custom_result = calculate_training_time(
    model_params=float(input("Enter model parameters (e.g., 26000000 for 26M): ")),
    tokens=float(input("Enter number of tokens (e.g., 100000000 for 100M): ")),
    gpu_tflops=float(input("Enter GPU TFLOPs (e.g., 312 for A100): ")),
    mfu=float(input("Enter MFU (typically 0.3-0.4): "))
)
print(f"\nEstimated training time: {custom_result['hours']:.2f} hours ({custom_result['days']:.2f} days)")

Example 1: MiniMind-26M (100M tokens)
Training time: 5.9 minutes (0.10 hours)
For 1B tokens: 1.0 hours

Example 2: MiniMind-109M (500M tokens)
Training time: 0.7 hours (0.0 days)

Custom Calculation:
--------------------------------------------------
Enter model parameters (e.g., 26000000 for 26M): 26000000
Enter number of tokens (e.g., 100000000 for 100M): 100000000
Enter GPU TFLOPs (e.g., 312 for A100): 312
Enter MFU (typically 0.3-0.4): 0.4

Estimated training time: 0.03 hours (0.00 days)


---
## 2. Training Cost Calculator

In [ ]:
def calculate_training_cost(training_hours, provider='runpod', gpu_type='a100_40gb'):
    """
    Calculate training cost based on provider and GPU type.

    Args:
        training_hours: Estimated training hours
        provider: Cloud provider name
        gpu_type: Type of GPU

    Returns:
        dict: Cost breakdown
    """
    # Pricing data (per hour)
    pricing = {
        'runpod': {
            'rtx_4090': 0.69,
            'a40': 0.79,
            'a100_40gb': 1.69,
            'a100_80gb': 2.89
        },
        'lambda': {
            'a100_40gb': 1.29,
            'a100_80gb': 2.49
        },
        'vast.ai': {
            'rtx_3090': 0.29,
            'a100_40gb': 0.89
        },
        'paperspace': {
            'p4000': 0.51,
            'a100_80gb': 3.09
        }
    }

    if provider not in pricing or gpu_type not in pricing[provider]:
        return None

    hourly_rate = pricing[provider][gpu_type]
    total_cost = training_hours * hourly_rate

    return {
        'provider': provider,
        'gpu_type': gpu_type,
        'hourly_rate': hourly_rate,
        'training_hours': training_hours,
        'total_cost': total_cost
    }

# Example usage
training_hours = 100

print(f"Cost comparison for {training_hours} hours of training:\n")
print(f"{'Provider':<15} {'GPU':<15} {'$/hour':<10} {'Total Cost':<15}")
print("-" * 55)

comparisons = [
    ('runpod', 'rtx_4090'),
    ('runpod', 'a100_40gb'),
    ('lambda', 'a100_40gb'),
    ('vast.ai', 'a100_40gb'),
]

for provider, gpu in comparisons:
    result = calculate_training_cost(training_hours, provider, gpu)
    if result:
        print(f"{provider:<15} {gpu:<15} ${result['hourly_rate']:<9.2f} ${result['total_cost']:<14.2f}")

Cost comparison for 100 hours of training:

Provider        GPU             $/hour     Total Cost     
-------------------------------------------------------
runpod          rtx_4090        $0.69      $69.00         
runpod          a100_40gb       $1.69      $169.00        
lambda          a100_40gb       $1.29      $129.00        
vast.ai         a100_40gb       $0.89      $89.00         


---
## 3. Memory Requirements Calculator

In [ ]:
def calculate_model_memory(params, precision='fp16'):
    """
    Calculate model memory requirements.

    Args:
        params: Number of parameters
        precision: 'fp32', 'fp16', 'int8', or 'int4'

    Returns:
        Memory in GB
    """
    bytes_per_param = {
        'fp32': 4,
        'fp16': 2,
        'int8': 1,
        'int4': 0.5
    }

    bytes_total = params * bytes_per_param[precision]
    return bytes_total / (1024**3)  # Convert to GB

def calculate_training_memory(params, batch_size=8, sequence_length=512):
    """
    Estimate total training memory (rough approximation).

    Formula: Model (fp16) + Optimizer States (2x) + Gradients (1x) + Activations
    """
    model_memory = calculate_model_memory(params, 'fp16')
    optimizer_memory = model_memory * 2  # Adam optimizer states
    gradient_memory = model_memory

    # Rough activation memory estimation (varies with architecture)
    activation_memory = (batch_size * sequence_length * params * 2) / (1024**3) / 1000

    total = model_memory + optimizer_memory + gradient_memory + activation_memory

    return {
        'model': model_memory,
        'optimizer': optimizer_memory,
        'gradients': gradient_memory,
        'activations': activation_memory,
        'total': total
    }

# Example calculations
models = [
    ('MiniMind-26M', 26e6),
    ('MiniMind-109M', 109e6),
    ('MiniMind-218M', 218e6),
    ('1B Model', 1e9),
    ('7B Model', 7e9)
]

print("Model Memory Requirements (Inference):\n")
print(f"{'Model':<20} {'FP32':<12} {'FP16':<12} {'INT8':<12} {'INT4':<12}")
print("-" * 68)

for name, params in models:
    fp32 = calculate_model_memory(params, 'fp32')
    fp16 = calculate_model_memory(params, 'fp16')
    int8 = calculate_model_memory(params, 'int8')
    int4 = calculate_model_memory(params, 'int4')
    print(f"{name:<20} {fp32:>8.2f} GB  {fp16:>8.2f} GB  {int8:>8.2f} GB  {int4:>8.2f} GB")

print("\n\nTraining Memory Requirements (Batch Size = 8):\n")
print(f"{'Model':<20} {'Model':<10} {'Optimizer':<12} {'Gradients':<12} {'Total':<12} {'Recommended GPU'}")
print("-" * 90)

for name, params in models:
    mem = calculate_training_memory(params, batch_size=8)

    # GPU recommendation
    if mem['total'] < 12:
        gpu = "RTX 3060 (12GB)"
    elif mem['total'] < 24:
        gpu = "RTX 4090 (24GB)"
    elif mem['total'] < 40:
        gpu = "A100 40GB"
    elif mem['total'] < 80:
        gpu = "A100 80GB"
    else:
        gpu = "Multiple A100s"

    print(f"{name:<20} {mem['model']:>6.2f} GB  {mem['optimizer']:>8.2f} GB  {mem['gradients']:>8.2f} GB  {mem['total']:>8.2f} GB  {gpu}")

Model Memory Requirements (Inference):

Model                FP32         FP16         INT8         INT4        
--------------------------------------------------------------------
MiniMind-26M             0.10 GB      0.05 GB      0.02 GB      0.01 GB
MiniMind-109M            0.41 GB      0.20 GB      0.10 GB      0.05 GB
MiniMind-218M            0.81 GB      0.41 GB      0.20 GB      0.10 GB
1B Model                 3.73 GB      1.86 GB      0.93 GB      0.47 GB
7B Model                26.08 GB     13.04 GB      6.52 GB      3.26 GB


Training Memory Requirements (Batch Size = 8):

Model                Model      Optimizer    Gradients    Total        Recommended GPU
------------------------------------------------------------------------------------------
MiniMind-26M           0.05 GB      0.10 GB      0.05 GB      0.39 GB  RTX 3060 (12GB)
MiniMind-109M          0.20 GB      0.41 GB      0.20 GB      1.64 GB  RTX 3060 (12GB)
MiniMind-218M          0.41 GB      0.81 GB      0.41 GB

---
## 4. Data Validation & Debugging Tools

In [ ]:
import json

def validate_json_file(filepath):
    """
    Validate a JSON/JSONL file for common issues.
    """
    print(f"Validating: {filepath}\n")

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            # Try to load as regular JSON first
            try:
                data = json.load(f)
                print(f"✓ Valid JSON file")
                print(f"✓ Contains {len(data)} items")

                # Show sample
                if len(data) > 0:
                    print(f"\nSample item:")
                    print(json.dumps(data[0], indent=2, ensure_ascii=False)[:200] + "...")

                return True

            except json.JSONDecodeError:
                # Try JSONL format
                f.seek(0)
                lines = f.readlines()
                valid_lines = 0

                for i, line in enumerate(lines):
                    try:
                        json.loads(line)
                        valid_lines += 1
                    except json.JSONDecodeError as e:
                        print(f"✗ Line {i+1} invalid: {e}")
                        print(f"  Content: {line[:100]}...")

                if valid_lines == len(lines):
                    print(f"✓ Valid JSONL file")
                    print(f"✓ Contains {valid_lines} valid lines")
                    return True
                else:
                    print(f"✗ JSONL file has {len(lines) - valid_lines} invalid lines")
                    return False

    except FileNotFoundError:
        print(f"✗ File not found: {filepath}")
        return False
    except Exception as e:
        print(f"✗ Error reading file: {e}")
        return False

# Example usage
print("Data Validation Tool")
print("=" * 50)
print("\nUpload your data file and run:")
print("validate_json_file('/path/to/your/data.json')")
print("\nExample validation:")
print("-" * 50)

# Create a sample file for demonstration
sample_data = [
    {"instruction": "What is AI?", "output": "AI is artificial intelligence..."},
    {"instruction": "Explain ML", "output": "Machine learning is..."}
]

with open('/tmp/sample_data.json', 'w') as f:
    json.dump(sample_data, f)

validate_json_file('/tmp/sample_data.json')

Data Validation Tool

Upload your data file and run:
validate_json_file('/path/to/your/data.json')

Example validation:
--------------------------------------------------
Validating: /tmp/sample_data.json

✓ Valid JSON file
✓ Contains 2 items

Sample item:
{
  "instruction": "What is AI?",
  "output": "AI is artificial intelligence..."
}...


True

---
## 5. Gradient Accumulation Helper

In [ ]:
def calculate_gradient_accumulation(target_batch_size, max_batch_per_gpu, num_gpus=1):
    """
    Calculate gradient accumulation steps needed.

    Args:
        target_batch_size: Desired effective batch size
        max_batch_per_gpu: Maximum batch size that fits in GPU memory
        num_gpus: Number of GPUs available

    Returns:
        dict: Configuration details
    """
    total_batch_per_step = max_batch_per_gpu * num_gpus
    accumulation_steps = target_batch_size // total_batch_per_step

    # Adjust if not evenly divisible
    if target_batch_size % total_batch_per_step != 0:
        accumulation_steps += 1

    actual_batch_size = accumulation_steps * total_batch_per_step

    return {
        'batch_size_per_gpu': max_batch_per_gpu,
        'num_gpus': num_gpus,
        'gradient_accumulation_steps': accumulation_steps,
        'actual_effective_batch_size': actual_batch_size,
        'target_batch_size': target_batch_size
    }

# Example configurations
print("Gradient Accumulation Calculator\n")
print("Example: Want batch size 32, but GPU only fits 8\n")

configs = [
    (32, 8, 1),   # Single GPU
    (128, 16, 2), # 2 GPUs
    (256, 8, 4),  # 4 GPUs
]

for target, max_per_gpu, gpus in configs:
    result = calculate_gradient_accumulation(target, max_per_gpu, gpus)
    print(f"Target batch: {target}, Max per GPU: {max_per_gpu}, GPUs: {gpus}")
    print(f"  → Use gradient_accumulation_steps = {result['gradient_accumulation_steps']}")
    print(f"  → Actual effective batch size = {result['actual_effective_batch_size']}")
    print()

# PyTorch code example
print("\nPyTorch Implementation:")
print("-" * 50)
print("""
# Instead of batch_size=32 (causes OOM)
batch_size = 8
gradient_accumulation_steps = 4  # Effective batch = 32

optimizer.zero_grad()
for i, batch in enumerate(dataloader):
    loss = model(batch)
    loss = loss / gradient_accumulation_steps  # Scale loss
    loss.backward()

    if (i + 1) % gradient_accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
""")

Gradient Accumulation Calculator

Example: Want batch size 32, but GPU only fits 8

Target batch: 32, Max per GPU: 8, GPUs: 1
  → Use gradient_accumulation_steps = 4
  → Actual effective batch size = 32

Target batch: 128, Max per GPU: 16, GPUs: 2
  → Use gradient_accumulation_steps = 4
  → Actual effective batch size = 128

Target batch: 256, Max per GPU: 8, GPUs: 4
  → Use gradient_accumulation_steps = 8
  → Actual effective batch size = 256


PyTorch Implementation:
--------------------------------------------------

# Instead of batch_size=32 (causes OOM)
batch_size = 8
gradient_accumulation_steps = 4  # Effective batch = 32

optimizer.zero_grad()
for i, batch in enumerate(dataloader):
    loss = model(batch)
    loss = loss / gradient_accumulation_steps  # Scale loss
    loss.backward()
    
    if (i + 1) % gradient_accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()



---
## 6. Reference URLs - Datasets

### Pretraining Datasets

**English Corpora:**
- The Pile: https://pile.eleuther.ai/
- C4: https://huggingface.co/datasets/allenai/c4
- Wikipedia: https://huggingface.co/datasets/wikimedia/wikipedia
- OpenWebText: https://huggingface.co/datasets/Skylion007/openwebtext

**Chinese Corpora:**
- WuDaoCorpora: https://www.scidb.cn/en/detail?dataSetId=c6a3fe684227415a9db8e21bac4a15ab
- Chinese Wikipedia: https://huggingface.co/datasets/wikimedia/wikipedia
- CLUECorpus: https://github.com/CLUEbenchmark/CLUECorpus2020

**Multilingual Corpora:**
- mC4: https://huggingface.co/datasets/allenai/c4
- OSCAR: https://oscar-corpus.com/
- CC100: https://data.statmt.org/cc-100/

**Code Datasets:**
- The Stack: https://huggingface.co/datasets/bigcode/the-stack
- CodeParrot: https://huggingface.co/datasets/codeparrot/github-code

### Supervised Fine-Tuning Datasets

**Instruction Following:**
- Alpaca: https://github.com/tatsu-lab/stanford_alpaca
- Dolly 15K: https://huggingface.co/datasets/databricks/databricks-dolly-15k
- OpenAssistant: https://huggingface.co/datasets/OpenAssistant/oasst1
- ShareGPT: https://huggingface.co/datasets/shareAI/ShareGPT-Chinese-English-90k

**Domain-Specific:**

*Medical:*
- MedQA: https://huggingface.co/datasets/bigbio/med_qa
- PubMedQA: https://huggingface.co/datasets/pubmed_qa
- MedDialog: https://huggingface.co/datasets/bigbio/meddialog

*Code:*
- CodeAlpaca: https://huggingface.co/datasets/sahil2801/CodeAlpaca-20k
- CodeContests: https://huggingface.co/datasets/deepmind/code_contests
- APPS: https://github.com/hendrycks/apps

*Legal:*
- LegalBench: https://huggingface.co/datasets/nguha/legalbench
- MultiLegal Pile: https://huggingface.co/datasets/joelito/Multi_Legal_Pile

### Preference/Alignment Data

- Anthropic HH-RLHF: https://huggingface.co/datasets/Anthropic/hh-rlhf
- OpenAssistant RLHF: https://huggingface.co/datasets/OpenAssistant/oasst1
- WebGPT Comparisons: https://huggingface.co/datasets/openai/webgpt_comparisons

---
## 7. Reference URLs - Pretrained Models

### Small Models (<1B parameters)

- MiniMind-109M: https://github.com/jingyaogong/minimind
- GPT-2-124M: https://huggingface.co/gpt2
- DistilGPT-2: https://huggingface.co/distilgpt2
- TinyLlama-1.1B: https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
- Phi-1.5: https://huggingface.co/microsoft/phi-1_5
- Pythia-1B: https://huggingface.co/EleutherAI/pythia-1b

### Medium Models (1B-7B)

- Llama-2-7B: https://huggingface.co/meta-llama/Llama-2-7b-hf
- Mistral-7B: https://huggingface.co/mistralai/Mistral-7B-v0.1
- Phi-2: https://huggingface.co/microsoft/phi-2
- Gemma-2B: https://huggingface.co/google/gemma-2b
- Qwen-1.8B: https://huggingface.co/Qwen/Qwen-1_8B

### Specialized Models

- CodeLlama-7B: https://huggingface.co/codellama/CodeLlama-7b-hf
- Meditron-7B: https://huggingface.co/epfl-llm/meditron-7b
- WizardMath-7B: https://huggingface.co/WizardLM/WizardMath-7B-V1.0
- Falcon-7B-Instruct: https://huggingface.co/tiiuae/falcon-7b-instruct

---
## 8. Reference URLs - Training Frameworks & Tools

### Training Frameworks

**MiniMind (This book!):**
- Repository: https://github.com/jingyaogong/minimind
- Focus: Educational, from-scratch implementation

**nanoGPT (Andrej Karpathy):**
- Repository: https://github.com/karpathy/nanoGPT
- Focus: Minimal, clean GPT implementation

**Lit-GPT (Lightning AI):**
- Repository: https://github.com/Lightning-AI/lit-gpt
- Focus: Production-ready, multiple architectures

**GPT-NeoX (EleutherAI):**
- Repository: https://github.com/EleutherAI/gpt-neox
- Focus: Large-scale training

### Fine-Tuning Tools

**Axolotl:**
- Repository: https://github.com/OpenAccess-AI-Collective/axolotl
- Features: Comprehensive, many techniques

**LLaMA-Factory:**
- Repository: https://github.com/hiyouga/LLaMA-Factory
- Features: Easy framework, Web UI

**Hugging Face PEFT:**
- Repository: https://github.com/huggingface/peft
- Features: LoRA, prefix tuning, etc.

### Inference & Deployment

**llama.cpp:**
- Repository: https://github.com/ggerganov/llama.cpp
- Features: CPU inference, quantization

**vLLM:**
- Repository: https://github.com/vllm-project/vllm
- Features: Fast GPU inference

**Text Generation Inference (HF):**
- Repository: https://github.com/huggingface/text-generation-inference
- Features: Production serving, optimized

### Educational Resources

**Andrej Karpathy's Lectures:**
- Neural Networks: Zero to Hero series on YouTube
- GitHub repositories with code examples

**Fast.ai NLP Course:**
- Website: https://course.fast.ai
- Practical deep learning

**Hugging Face Course:**
- Website: https://huggingface.co/course
- Comprehensive NLP course

---
## 9. Cloud Provider Information

### Google Colab

- **Website:** https://colab.research.google.com
- **Free Tier:** T4 GPU, 16GB RAM, limited usage
- **Pro:** $9.99/mo - T4/P100, priority access
- **Pro+:** $49.99/mo - V100/A100, background execution
- **Education Program:** https://blog.google/outreach-initiatives/education/colab-higher-education/

### RunPod

- **Website:** https://www.runpod.io
- **Pricing:** https://www.runpod.io/pricing
- **RTX 4090:** $0.69/hr
- **A100 40GB:** $1.69/hr
- **A100 80GB:** $2.89/hr
- **Features:** Spot pricing, no commitment

### Lambda Labs

- **Website:** https://lambdalabs.com
- **Cloud:** https://lambdalabs.com/service/gpu-cloud
- **A100 40GB:** $1.29/hr
- **A100 80GB:** $2.49/hr
- **Features:** Reserved instances, fast networking

### Paperspace

- **Website:** https://www.paperspace.com
- **Gradient:** https://www.paperspace.com/gradient
- **Free GPU:** 6hrs/month
- **P4000:** $0.51/hr
- **A100 80GB:** $3.09/hr

### Vast.ai

- **Website:** https://vast.ai
- **Pricing:** https://vast.ai/pricing
- **RTX 3090:** ~$0.29/hr (spot market)
- **A100 40GB:** ~$0.89/hr (spot market)
- **Note:** Prices vary, can be interrupted

---
## 10. Quick Setup Commands for Colab

In [ ]:
# Check GPU availability
!nvidia-smi

# Install common packages
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install transformers datasets accelerate peft bitsandbytes

# Check PyTorch CUDA
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

---
## Additional Resources

**Book Repository:**
- Main Code: https://rewriting.ai/bookcode/slm.php
- MiniMind Project: https://github.com/jingyaogong/minimind

**Documentation:**
- PyTorch: https://pytorch.org/docs/
- Hugging Face Transformers: https://huggingface.co/docs/transformers
- Anthropic Claude API: https://docs.claude.com

---

**End of Chapter 13 Reference Notebook**

Remember to enable GPU in Colab: Runtime → Change runtime type → GPU